In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import r2_score, accuracy_score, classification_report
from sklearn.preprocessing import PolynomialFeatures
from xgboost import XGBClassifier
import joblib


df = pd.read_csv("/content/world_up_match_data.csv") #Loading in the dataset using the pandas library

#Determing the winner of the match based on the number of goals
def get_winner(row):
  if row["hgoals"] > row["agoals"] :
    return row["hname"]
  elif row["agoals"] > row["hgoals"] :
    return row["aname"]
  else:
    return "Draw"

#--------Setting up columns-----------------
df["winner"] = df.apply(get_winner, axis=1) #setting up a target field
team_strength = df.groupby("hname")["hgoals"].mean() #Getting the average number of goals scored by each team
df["home_strength"] = df["hname"].map(team_strength) #Setting up a feature field
df["away_strength"] = df["aname"].map(team_strength)

# Fill NaN values that might arise if a team in 'hname' or 'aname' doesn't exist in 'team_strength'
df.fillna(0, inplace=True)

#Encoding team names for better comparisons
df = pd.get_dummies(df, columns=["hname", "aname"])

x = df.drop(columns=["winner"]) #Setting up features to not include an text or string fields

print("Shape of x before PolynomialFeatures:", x.shape)
poly = PolynomialFeatures()
x = poly.fit_transform(x) #Increasing the number of features to make more effectient
print("Shape of x after PolynomialFeatures:", x.shape)

y = df["winner"] #Setting up the target field

#Splitting the data into training and test data by 20% and shuffling them systematically in a fixed way
x_train, x_test, y_train, y_test = train_test_split(x,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)

RFC = RandomForestClassifier(n_estimators=300,
                    max_depth=None,
                  random_state =42
  )
GBC = GradientBoostingClassifier(n_estimators=100, random_state =42) # Reduced n_estimators for faster training
xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss"
)
#xgb.fit(x_train, y_train)
#print("XGB:", xgb.score(x_test, y_test))

#for i in [RFC]: #Iterating through differnt models to compare
 # i.fit(x_train, y_train) #Fitting the model to the training data

#  y_pred = i.predict(x_test)

  #accuracy = accuracy_score(y_test, y_pred)
 # print(f"{i} Accuracy: {accuracy}")
  #print()
RFC.fit(x_train, y_train)
joblib.dump(RFC, 'Betting_Predictions_Draft1_Joblib')
y_pred = RFC.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest Accuracy: {accuracy}")

#report = classification_report(y_test, y_pred)
#print("Classification Report")
#print(report)

Shape of x before PolynomialFeatures: (1482, 264)
Shape of x after PolynomialFeatures: (1482, 35245)
Random Forest Accuracy: 0.6060606060606061


In [33]:
local_model = joblib.load("Betting_Predictions_Draft1_Joblib")
y_pred = local_model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest Accuracy: {accuracy}")

Random Forest Accuracy: 0.6060606060606061
